In [ ]:
SCOPE = {   'search_terms': [   'spectral-domain OCT deep learning',
                        'swept-source OCT classification',
                        'diabetic macular edema deep learning',
                        'age-related macular degeneration classification',
                        'choroidal neovascularization detection',
                        'central serous chorioretinopathy diagnosis',
                        'retinal vein occlusion classification',
                        'epiretinal membrane segmentation',
                        'macular hole detection',
                        'drusen segmentation']}

In [ ]:
from literature_ai.review_service.review.nodes.search_papers import search_papers
from literature_ai.review_service.review.nodes.compute_elbow import compute_elbow
from literature_ai.review_service.review.nodes.prune_results import prune_results
from literature_ai.review_service.review.nodes.deduplicate_papers import deduplicate_papers
from literature_ai.review_service.review.nodes.abstract_review import abstract_review

import requests
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
BASE_URL = "http://localhost:8000"
RESULTS_PER_TERM = 1000
EMBEDDING_MODELS = ["specter_v2", "specter_v1", "bge_m3", "qwen3_embedding_4b", "text-embedding-3-small"]

response = requests.get(f"{BASE_URL}/embedding-models")
response.raise_for_status()
runs = response.json()["runs"]

MODEL_RUN_IDS = {}
for run in runs:
    MODEL_RUN_IDS.setdefault(run["embedding_model"], run["run_id"])

MODEL_RUN_IDS = {model: MODEL_RUN_IDS[model] for model in EMBEDDING_MODELS}

In [ ]:
search_terms = SCOPE["search_terms"]

paper_rows = []
elbow_rows = []

for model_name in EMBEDDING_MODELS:
    run_id = MODEL_RUN_IDS[model_name]
    for term in search_terms:
        state = {
            "search_terms": [term],
            "term_index": 0,
            "embedding_run_id": run_id,
            "results_per_term": RESULTS_PER_TERM,
        }
        state.update(search_papers(state))
        state.update(compute_elbow(state))

        for result in state["current_term_results"]:
            paper_rows.append(
                {
                    "embedding_model": model_name,
                    "search_term": term,
                    "paper_id": result["paper_id"],
                    "rank": result["rank"],
                    "similarity": result["similarity"],
                }
            )

        elbow_rows.append(
            {
                "embedding_model": model_name,
                "search_term": term,
                "elbow_rank": state["current_term_elbow_rank"],
            }
        )

In [ ]:
papers_long = pd.DataFrame(paper_rows)

papers_wide = papers_long.pivot_table(
    index=["search_term", "paper_id"],
    columns="embedding_model",
    values=["rank", "similarity"],
    aggfunc="first",
)

elbows_df = pd.DataFrame(elbow_rows).pivot_table(
    index="search_term",
    columns="embedding_model",
    values="elbow_rank",
    aggfunc="first",
)

papers_wide

In [ ]:
def plot_term_elbows(term):
    subset = papers_long[papers_long["search_term"] == term]

    fig, ax = plt.subplots(figsize=(8, 5))
    for model_name, group in subset.groupby("embedding_model"):
        group = group.sort_values("rank")
        ax.plot(group["rank"], group["similarity"], label=model_name)

        elbow_rank = elbows_df.loc[term, model_name]
        if pd.notna(elbow_rank):
            elbow_point = group[group["rank"] == elbow_rank]
            if not elbow_point.empty:
                ax.scatter(elbow_point["rank"], elbow_point["similarity"], zorder=5)

    ax.set_xlabel("rank")
    ax.set_ylabel("similarity")
    ax.set_title(term)
    ax.legend()
    return fig


plot_term_elbows(search_terms[0])